# The leaked value the copy detector caught

The leaked arm (defect D2) shows sample rows in the schema block, the answer row among them, on the share of prompts where such a row exists. A policy that learns to copy the value from the prompt instead of querying for it is paid by the honest comparator and fails on every twin. This notebook draws the share of answers the copy detector flagged at every step, says how many steps it took to fire, and quotes three flagged answers.

In [ ]:
import json
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from goodhart_evaluation import arms

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
runs = arms.arm_runs(ROOT)
print("promoted defect study runs:", runs or "none yet")

In [ ]:
run = runs.get("leaked")
if run is None:
    print("the leaked arm has not run yet")
else:
    rows = arms.monitor(ROOT, run)
    shares = [(int(r["step"]), float(r.get("share_copied_value", 0.0))) for r in rows]
    fired = arms.first_alert(ROOT, run, "share_copied_value")
    print("steps watched:", len(shares), "| the copy detector fired at step:", fired)
    flagged = [r for r in arms.rollouts(ROOT, run) if r.get("copied_value")][:3]
    for r in flagged:
        print(f"step {r['step']}, {r['item_id']}: {r['text'][:200]!r}")